# WorldSplat-Prune — front-end smoke test (caption → GroundingDINO → SAM2 → instance maps)

Exercises the **prompt-driven labelling front-end** (§2.2) end to end on one scene
and stops at the **instance-label map** output — the post-reconstruction pruning
operator is not involved.

| stage | module | output |
|---|---|---|
| 1 | `frontend.caption` | seed phrase list (`phrases_seed.txt`) |
| 2 | `frontend.grounding` | per-frame boxes (`detections.json`) |
| 3 | `frontend.sam2_box` | per-class instance maps (`semantic_instance_<cls>/`) |
| 4 | `prep.fuse_instances` | fused per-class ids + composite maps (`semantic_composite/`) |

Every stage is skippable and its intermediate artifact is kept on disk, so a
failure at one stage doesn't cost the earlier work. Runs on a Colab GPU (a T4 is
enough with the `-tiny` detector and a subsampled `STRIDE`).

> **Does this work at the current state?** Yes — the front-end modules exist
> (`worldsplat_prune/frontend/`, `worldsplat_prune/datasets/composite.py`,
> `worldsplat_prune/prep/fuse_instances.fuse_instances_per_class`). Their **pure**
> parts are unit-tested; the **model** paths (`torch` / `transformers` / `sam2`)
> are exercised for the first time here, so treat this as the smoke test.


## 1. Mount Google Drive

Skip this cell if you run locally / already have the dataset on disk.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Get this repo + front-end deps

Colab already ships PyTorch. We add `transformers` + `timm` (GroundingDINO and the
captioner) and install SAM2 from source. The CUDA rasterizer submodules are **not**
needed for labelling, but we init them so the same checkout can train afterwards.


In [ ]:
import os

REPO_URL = "https://github.com/satyaba/worldsplat-prune.git"
REPO_DIR = "/content/worldsplat-prune"
BRANCH = "main"

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch --all -q
    !git -C {REPO_DIR} checkout {BRANCH} -q
    !git -C {REPO_DIR} pull --ff-only -q

os.chdir(REPO_DIR)

!git submodule update --init --recursive submodules/diff-gaussian-rasterization submodules/simple-knn

!pip install -q plyfile opencv-python joblib
!pip install -q transformers timm
!pip install -q "git+https://github.com/facebookresearch/sam2.git"

print("cwd:", os.getcwd())

## 3. GPU check

GroundingDINO and SAM2 both run on GPU; CPU works but is slow. The models are small
(`grounding-dino-tiny`, `sam2.1_hiera_large`) so a T4 suffices.


In [ ]:
!nvidia-smi

import torch
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Configure paths


In [ ]:
import os
from pathlib import Path

HOME_DIR = "/content/drive/MyDrive/Magister/Thesis/Script"
DATASET_NAME = "office_0/Sequence_1"          # e.g. "mipnerf360/garden"

os.environ["HOME_DIR"] = HOME_DIR
DATASET_ROOT = Path(HOME_DIR) / "Dataset"
os.environ["DATASET_ROOT"] = str(DATASET_ROOT)

# Resolve the scene. Prepared scenes live in Dataset/<name>; raw Mip-NeRF 360
# scenes retrieved by Retrieve_Datasets.ipynb live in $HOME_DIR/raw/<scene>
# (their zip already ships images/ + sparse/0). Add your own path below if your
# layout differs. The first candidate that has an images/ dir wins.
CANDIDATES = [
    DATASET_ROOT / DATASET_NAME,                        # prepared
    Path(HOME_DIR) / "raw" / Path(DATASET_NAME).name,   # raw retrieval
    Path(DATASET_NAME),                                 # explicit path
]
DATASET_DIR = next((c for c in CANDIDATES if (c / "images").is_dir()), None)
if DATASET_DIR is None:
    raise FileNotFoundError(
        "no 'images/' dir found. Edit DATASET_NAME, or add a candidate:\n  "
        + "\n  ".join(str(c) for c in CANDIDATES))
print("DATASET_DIR :", DATASET_DIR)

n_images = len([p for p in (DATASET_DIR / "images").glob("*") if p.is_file()])
print("images      :", n_images)
print("colmap      :", (DATASET_DIR / "sparse" / "0" / "cameras.bin").exists()
      or (DATASET_DIR / "sparse" / "0" / "cameras.txt").exists())


## 5. Phrase list — seed (optional), then curate

D15: an offline captioner **seeds** the phrase list and a human **curates** it.
Run **5a** to seed; then edit `PHRASES` in **5b**. If you'd rather skip the
captioner, just set `PHRASES` by hand and run 5b only.


In [ ]:
# 5a — seed phrases from frame captions (optional)
from worldsplat_prune.frontend.caption import HFCaptioner, seed_phrase_list

captioner = HFCaptioner("Salesforce/blip-image-captioning-base", device="cuda")
seed = seed_phrase_list(DATASET_DIR, captioner, n=12,
                        out=str(DATASET_DIR / "phrases_seed.txt"))
print(f"{len(seed)} seed phrases -> curate them in 5b")
seed[:40]

In [ ]:
# 5b — curated phrase list (edit me). Paste/trim the seed above or type your own.
PHRASES = [
    "wall", "floor", "ceiling", "table", "desk", "chair", "sofa",
    "cabinet", "shelf", "monitor", "door", "window", "whiteboard", "lamp",
]
print(len(PHRASES), "phrases")
PHRASES

## 6. GroundingDINO — phrases → boxes

Open-vocabulary detection: each phrase becomes zero or more boxes per frame. `STRIDE`
subsamples frames for the smoke test (set `1` for the full scene).


In [ ]:
from worldsplat_prune.frontend.grounding import detect_dataset

STRIDE = 20        # label every 20th frame for the test; 1 = full scene
RESIZE = 1024      # cap inference resolution to save memory; None = native

detections = detect_dataset(
    DATASET_DIR, PHRASES, "detections.json",
    device="cuda", stride=STRIDE, resize_long_side=RESIZE,
    box_threshold=0.30, text_threshold=0.25,
)

import collections
counts = collections.Counter()
for det in detections.values():
    for ph, boxes in det.items():
        counts[ph] += len(boxes)
print(f"{len(detections)} frames processed")
for ph, c in counts.most_common():
    print(f"  {ph:16s} {c}")

## 7. SAM2 box-prompt — boxes → per-class instance maps

Downloads SAM2.1-large once, then turns each box into a mask and writes one
instance map **per class** at native resolution: `semantic_instance_<slug>/<stem>.png`.
`stride=1` here (we only label the frames that have detections).


In [ ]:
from pathlib import Path
from worldsplat_prune.frontend.sam2_box import label_dataset_boxes

CKPT_DIR = Path(HOME_DIR) / "checkpoints"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
SAM2_CKPT = CKPT_DIR / "sam2.1_hiera_large.pt"
if not SAM2_CKPT.exists():
    !wget -q --show-progress -O "{SAM2_CKPT}" https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

class_dirs = label_dataset_boxes(
    DATASET_DIR, detections,
    sam2_ckpt=str(SAM2_CKPT), device="cuda",
    min_area=100, max_area_frac=0.9, fill_void=True,
    resize_long_side=RESIZE, stride=1, overwrite=True,
)
class_dirs

## 8. Per-class fusion → composite instance-label map

Fuses each class independently (association never crosses classes) and composes the
canonical composite id `class_id * 10000 + instance_id` — the `L_v` the pruning
operator consumes. Class ids follow the curated `PHRASES` order.


In [ ]:
from worldsplat_prune.prep.fuse_instances import fuse_instances_per_class

class_ids = {p: i + 1 for i, p in enumerate(PHRASES)}
summary = fuse_instances_per_class(
    DATASET_DIR, class_dirs, class_ids=class_ids,
    metric="jaccard", min_score=0.25, min_views=2,
)
{ "composite_views": summary["composite_views"],
  "composite_points": summary["composite_points"],
  "class_ids": summary["class_ids"] }

## 9. Inspect the composite maps


In [ ]:
from pathlib import Path
from worldsplat_prune.datasets import get_adapter
from worldsplat_prune.datasets.composite import read_composite_map, decode, COMPOSITE_STRIDE

adapter = get_adapter("composite", DATASET_DIR)
print(adapter.meta())

comp_files = sorted((Path(DATASET_DIR) / "semantic_composite").glob("*.npy"))
print(len(comp_files), "composite maps")

if comp_files:
    ids = read_composite_map(comp_files[0])
    cls, inst = decode(ids)
    print("labeled px:", int((ids > 0).sum()),
          "| classes:", sorted(set(cls[cls > 0].tolist())),
          "| instances:", len({int(i) for i in inst[inst > 0].tolist()}))

In [ ]:
from pathlib import Path
from worldsplat_prune.datasets import get_adapter
from worldsplat_prune.datasets.composite import read_composite_map, decode, COMPOSITE_STRIDE

adapter = get_adapter("composite", DATASET_DIR)
print(adapter.meta())

# per-view composite maps only: exclude composite_point_labels.npy (the per-point
# sidecar written next to them).
comp_dir = Path(DATASET_DIR) / "semantic_composite"
comp_files = sorted(p for p in comp_dir.glob("*.npy")
                    if not p.stem.endswith("point_labels"))
print(len(comp_files), "per-view composite maps")

if comp_files:
    ids = read_composite_map(comp_files[0])
    cls, inst = decode(ids)
    print("file:", comp_files[0].name)
    print("labeled px:", int((ids > 0).sum()),
          "| classes:", sorted({int(c) for c in cls[cls > 0].tolist()}),
          "| instances:", len({int(i) for i in inst[inst > 0].tolist()}))


import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image

img_dir = Path(DATASET_DIR) / "images"
img_index = {p.stem: p for p in img_dir.iterdir() if p.is_file()}


def show_composite(path):
    ids = read_composite_map(path)
    cls, inst = decode(ids)
    stem = path.stem
    rgb = None
    if stem in img_index:
        rgb = np.array(Image.open(img_index[stem]).convert("RGB"))
        if cls.shape != rgb.shape[:2]:       # map may be at COLMAP camera resolution
            cls = cv2.resize(cls, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)
            inst = cv2.resize(inst, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)

    panels = []
    if rgb is not None:
        panels.append((rgb, "RGB — " + stem, None))
    panels.append((np.where(cls > 0, cls, np.nan), "class id", "tab20"))
    panels.append((np.where(inst > 0, inst, np.nan), "instance id", "nipy_spectral"))

    fig, axes = plt.subplots(1, len(panels), figsize=(6 * len(panels), 5))
    for ax, (im, title, cmap) in zip(np.atleast_1d(axes), panels):
        ax.imshow(im, cmap=cmap, interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


if comp_files:
    pick = next((p for p in comp_files if p.stem in img_index), comp_files[0])
    if pick.stem not in img_index:
        print("note: no RGB file matches the map stems; showing label maps only")
        print("  map stems   :", [p.stem for p in comp_files[:5]])
        print("  image stems :", list(img_index)[:5])
    show_composite(pick)
else:
    print("no per-view composite maps — run the fusion cell (section 8) first")


In [ ]:
# from worldsplat_prune.prep.build_semantic_maps import build_semantic_maps
# build_semantic_maps(DATASET_DIR, PHRASES, sam2_ckpt=str(SAM2_CKPT), stride=1)